In [ ]:
import pandas as pd
import numpy as np

In [ ]:
df=pd.read_csv("output(42) (4).csv")

In [ ]:
df.head()

,Unnamed: 0,sample_id,image_link,price,Item Name,Value,Unit,Bullet Point 1
0,0,33127,https://m.media-amazon.com/images/I/51mo8htwTH...,4.89,"La Victoria Green Taco Sauce Mild, 12 Ounce (P...",72.00,Fl Oz,NaN
1,1,198967,https://m.media-amazon.com/images/I/71YtriIHAA...,13.12,"Salerno Cookies, The Original Butter Cookies, ...",32.00,Ounce,original butter cookies : classic butter cooki...
2,2,261251,https://m.media-amazon.com/images/I/51+PFEe-w-...,1.97,"Bear Creek Hearty Soup Bowl, Creamy Chicken wi...",11.40,Ounce,loaded hearty long grain wild rice vegetables ...
3,3,55858,https://m.media-amazon.com/images/I/41mu0HAToD...,30.34,Judee’s Blue Cheese Powder 11.25 oz - Gluten-F...,11.25,Ounce,"add favorite appetizers , dips & spreads . use..."
4,4,292686,https://m.media-amazon.com/images/I/41sA037+Qv...,66.49,"kedem Sherry Cooking Wine, 12.7 Ounce - 12 per...",12.00,Count,"kedem sherry cooking wine , 12.7 ounce - 12 pe..."


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75000 entries, 0 to 74999
Data columns (total 8 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Unnamed: 0      75000 non-null  int64  
 1   sample_id       75000 non-null  int64  
 2   image_link      75000 non-null  object 
 3   price           75000 non-null  float64
 4   Item Name       74992 non-null  object 
 5   Value           74060 non-null  float64
 6   Unit            74059 non-null  object 
 7   Bullet Point 1  64967 non-null  object 
dtypes: float64(2), int64(2), object(4)
memory usage: 4.6+ MB


In [ ]:
# ==============================================
# Step 1 – Create and Save TF-IDF Embeddings
# ==============================================

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
import re, string
import joblib

# Load data
df = pd.read_csv('output(42) (4).csv')

# Columns: ['sample_id', 'image_link', 'price', 'Item Name', 'Value', 'Unit', 'Bullet Point 1']
text_cols = ['Item Name', 'Bullet Point 1', 'Value', 'Unit']
for col in text_cols:
    df[col] = df[col].fillna('')

# Combine all textual info
df['text'] = (
    df['Item Name'].astype(str) + ' ' +
    df['Bullet Point 1'].astype(str) + ' ' +
    df['Value'].astype(str) + ' ' +
    df['Unit'].astype(str)
)

# Simple clean-up
def clean_text(s):
    s = s.lower()
    s = re.sub(r'https?://\S+|www\.\S+', '', s)
    s = re.sub(f"[{re.escape(string.punctuation)}]", ' ', s)
    s = re.sub(r'\s+', ' ', s)
    return s.strip()

df['text'] = df['text'].apply(clean_text)

# Vectorize text → TF-IDF
vectorizer = TfidfVectorizer(
    max_features=20000,
    stop_words='english',
    ngram_range=(1, 2)
)

X = vectorizer.fit_transform(df['text'])
y = df['price'].values

print(f"TF-IDF matrix shape: {X.shape}")

# Save everything for next step
joblib.dump(X, '/content/tfidf_embeddings.pkl')
joblib.dump(y, '/content/target_price.pkl')
joblib.dump(vectorizer, '/content/tfidf_vectorizer.pkl')

print("✅ Saved TF-IDF embeddings and vectorizer to /content/")


TF-IDF matrix shape: (75000, 20000)
✅ Saved TF-IDF embeddings and vectorizer to /content/


In [ ]:
# ==============================================
# Step 2 – Model Training with SMAPE Metric
# ==============================================

import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor

# Load saved data
X = joblib.load('/content/tfidf_embeddings.pkl')
y = joblib.load('/content/target_price.pkl')

# Split data
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Define SMAPE
def smape(y_true, y_pred):
    numerator = np.abs(y_true - y_pred)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    smape_val = np.mean(numerator / np.maximum(denominator, 1e-8)) * 100
    return smape_val

# Train model
xgb = XGBRegressor(
    n_estimators=500,
    learning_rate=0.1,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    tree_method='hist'
)
xgb.fit(X_train, y_train)

# Predict and evaluate
y_pred = xgb.predict(X_val)
score = smape(y_val, y_pred)
print(f"✅ Validation sMAPE: {score:.2f}%")


✅ Validation sMAPE: 61.58%


In [ ]:
import joblib

# Save model
joblib.dump(xgb, '/content/xgb_model.pkl')

# If you used TF-IDF vectorizer
joblib.dump(vectorizer, '/content/tfidf_vectorizer.pkl')

print("✅ Model and vectorizer saved to /content/")


✅ Model and vectorizer saved to /content/


In [ ]:
!pip install -q transformers datasets torch accelerate
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import BertTokenizer, BertModel, Trainer, TrainingArguments
from datasets import Dataset
import pandas as pd
import numpy as np


In [ ]:
# df should already be loaded
df['text'] = (
    df['Item Name'].fillna('') + ' ' +
    df['Bullet Point 1'].fillna('')
)
df = df[['text', 'price']].dropna()


In [ ]:
df.head()

,text,price
0,"La Victoria Green Taco Sauce Mild, 12 Ounce (P...",4.89
1,"Salerno Cookies, The Original Butter Cookies, ...",13.12
2,"Bear Creek Hearty Soup Bowl, Creamy Chicken wi...",1.97
3,Judee’s Blue Cheese Powder 11.25 oz - Gluten-F...,30.34
4,"kedem Sherry Cooking Wine, 12.7 Ounce - 12 per...",66.49


In [ ]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

def tokenize(batch):
    return tokenizer(
        batch['text'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

dataset = Dataset.from_pandas(df)
dataset = dataset.map(tokenize, batched=True)
dataset.set_format(
    type='torch',
    columns=['input_ids', 'attention_mask', 'price']
)

train_test = dataset.train_test_split(test_size=0.1, seed=42)
train_dataset = train_test['train']
val_dataset = train_test['test']


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Map:   0%|          | 0/75000 [00:00<?, ? examples/s]

In [ ]:
class BertRegressor(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = BertModel.from_pretrained('bert-base-uncased')
        self.dropout = nn.Dropout(0.2)
        self.regressor = nn.Linear(self.bert.config.hidden_size, 1)

    def forward(self, input_ids, attention_mask, labels=None):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # CLS token
        x = self.dropout(cls_embedding)
        preds = self.regressor(x).squeeze(-1)

        loss = None
        if labels is not None:
            loss_fn = nn.L1Loss()  # MAE loss
            loss = loss_fn(preds, labels)
        return {"loss": loss, "logits": preds}


In [ ]:
import transformers, datasets
print("transformers", transformers.__version__)
print("datasets", datasets.__version__)


transformers 4.57.0
datasets 4.0.0


In [ ]:
# upgrade transformers, datasets, accelerate, and tokenizers
!pip install -q --upgrade transformers datasets accelerate tokenizers
# (optional) install a specific version if you prefer, e.g. !pip install transformers==4.34.0


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.3/506.3 kB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.8/42.8 MB 13.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pylibcudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.
cudf-cu12 25.6.0 requires pyarrow<20.0.0a0,>=14.0.0; platform_machine == "x86_64", but you have pyarrow 21.0.0 which is incompatible.


In [ ]:
import inspect
from transformers import TrainingArguments

print(inspect.signature(TrainingArguments.__init__))


(self, output_dir: Optional[str] = None, overwrite_output_dir: bool = False, do_train: bool = False, do_eval: bool = False, do_predict: bool = False, eval_strategy: Union[transformers.trainer_utils.IntervalStrategy, str] = 'no', prediction_loss_only: bool = False, per_device_train_batch_size: int = 8, per_device_eval_batch_size: int = 8, per_gpu_train_batch_size: Optional[int] = None, per_gpu_eval_batch_size: Optional[int] = None, gradient_accumulation_steps: int = 1, eval_accumulation_steps: Optional[int] = None, eval_delay: float = 0, torch_empty_cache_steps: Optional[int] = None, learning_rate: float = 5e-05, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, max_grad_norm: float = 1.0, num_train_epochs: float = 3.0, max_steps: int = -1, lr_scheduler_type: Union[transformers.trainer_utils.SchedulerType, str] = 'linear', lr_scheduler_kwargs: Union[dict[str, Any], str] = <factory>, warmup_ratio: float = 0.0, warmup_steps: int = 

In [ ]:
from transformers import Trainer, TrainingArguments, AutoTokenizer, AutoConfig, AutoModelForSequenceClassification
from datasets import Dataset
import numpy as np
import torch

# Ensure df has 'text' and 'price' columns
# df['text'] = df['Item Name'].fillna('') + ' ' + df['Bullet Point 1'].fillna('')
# df = df[['text', 'price']].dropna().reset_index(drop=True)

# Tokenizer
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def preprocess(examples):
    return tokenizer(examples['text'], truncation=True, padding='max_length', max_length=128)

# Create dataset
hf_ds = Dataset.from_pandas(df)
hf_ds = hf_ds.map(preprocess, batched=True)
hf_ds = hf_ds.rename_column("price", "labels")
hf_ds.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])

# Train/validation split
split = hf_ds.train_test_split(test_size=0.1, seed=42)
train_ds = split['train']
val_ds = split['test']

# Model setup for regression
config = AutoConfig.from_pretrained(model_name, num_labels=1)
model = AutoModelForSequenceClassification.from_pretrained(model_name, config=config)

# ✅ Updated TrainingArguments for your version
training_args = TrainingArguments(
    output_dir="bert-price",
    num_train_epochs=2,
    per_device_train_batch_size=8,
    eval_strategy="epoch",        # use epoch for both
    save_strategy="epoch",        # match eval
    load_best_model_at_end=True,
    metric_for_best_model="loss",
    greater_is_better=False,
)


# SMAPE metric
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    preds = preds.flatten()
    labels = labels.flatten()
    smape = 100 * np.mean(
        2 * np.abs(preds - labels) / (np.abs(preds) + np.abs(labels) + 1e-8)
    )
    return {"smape": smape}

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics
)

# Train model
trainer.train()

# Evaluate
metrics = trainer.evaluate()
print(metrics)


Map:   0%|          | 0/75000 [00:00<?, ? examples/s]

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: shashankyadavriiii (shashankyadavriiii-indian-institute-of-technology) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Epoch,Training Loss,Validation Loss,Smape
1,438.997000,705.183716,53.601543
2,552.348600,625.288940,50.942047


{'eval_loss': 625.2889404296875, 'eval_smape': 50.942047119140625, 'eval_runtime': 26.2613, 'eval_samples_per_second': 285.591, 'eval_steps_per_second': 35.718, 'epoch': 2.0}


In [ ]:
test_df=pd.read_csv("output_text3.csv")

In [ ]:
test_df.head()

,Unnamed: 0,sample_id,image_link,Item Name,Value,Unit,important_words
0,0,100179,https://m.media-amazon.com/images/I/71hoAn78AW...,Rani 14-Spice Eshamaya's Mango Chutney (Indian...,10.5,ounce,chutney mango rani chilli powder
1,1,245611,https://m.media-amazon.com/images/I/61ex8NHCIj...,Natural MILK TEA Flavoring extract by HALO PAN...,2.0,fl oz,halo asianinspired recipes exclusively pantry
2,2,146263,https://m.media-amazon.com/images/I/61KCM61J8e...,Honey Filled Hard Candy - Bulk Pack 2 Pounds -...,32.0,ounce,honey candy hard soothing filled
3,3,95658,https://m.media-amazon.com/images/I/51Ex6uOH7y...,Vlasic Snack'mm's Kosher Dill 16 Oz (Pack of 2),2.0,count,NaN
4,4,36806,https://m.media-amazon.com/images/I/71QYlrOMoS...,"McCormick Culinary Vanilla Extract, 32 fl oz -...",32.0,fl oz,vanilla extract mccormick pure culinary


In [ ]:
test_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 75000 entries, 0 to 74999
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Unnamed: 0       75000 non-null  int64  
 1   sample_id        75000 non-null  int64  
 2   image_link       75000 non-null  object 
 3   Item Name        74992 non-null  object 
 4   Value            75000 non-null  float64
 5   Unit             75000 non-null  object 
 6   important_words  64854 non-null  object 
dtypes: float64(1), int64(2), object(4)
memory usage: 4.0+ MB


In [ ]:
test_text = (
    test_df[['Item Name', 'important_words', 'Unit']]
    .fillna('')
    .agg(' '.join, axis=1)
)
X_test_tfidf = vectorizer.transform(test_text)


In [ ]:
import joblib

# Load your trained model
xgb_model = joblib.load("/content/xgb_model.pkl")

print("✅ Model loaded successfully!")


✅ Model loaded successfully!


In [ ]:
vectorizer = joblib.load("/content/tfidf_vectorizer.pkl")
print("✅ Vectorizer loaded successfully!")


✅ Vectorizer loaded successfully!


In [ ]:
# Transform test text to TF-IDF features
X_test_tfidf = vectorizer.transform(test_text)

# Predict prices
test_preds = xgb_model.predict(X_test_tfidf)


In [ ]:
submission_xgb = pd.DataFrame({
    'sample_id': test_df['sample_id'],
    'price': test_preds
})

submission_xgb.to_csv("submission_xgb.csv", index=False)
print("✅ Submission file saved as submission_xgb.csv")


✅ Submission file saved as submission_xgb.csv


In [ ]:
import os
os.listdir("/content/bert-price")


['runs', 'checkpoint-8438', 'checkpoint-16876']

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import os

# Folder where your trained weights are
model_dir = os.path.join("/content/bert-price", "checkpoint-16876")

# ✅ Load tokenizer from the base pretrained model
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

# ✅ Load your fine-tuned model weights
model = AutoModelForSequenceClassification.from_pretrained(model_dir)
model.eval()

print("✅ Model and tokenizer loaded successfully!")


✅ Model and tokenizer loaded successfully!


In [ ]:
import torch
from tqdm import tqdm

# Assuming your test data is already in `test_df`
test_df = test_df.fillna("")

# Combine key textual columns
test_texts = (
    test_df["Item Name"].astype(str) + " " + test_df["important_words"].astype(str)
).tolist()


In [ ]:
# Tokenize in batches
inputs = tokenizer(
    test_texts,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors="pt"
)


In [ ]:
# Send model to device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Predict in batches
batch_size = 64
predictions = []

for i in tqdm(range(0, len(test_texts), batch_size)):
    batch = {k: v[i:i+batch_size].to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**batch)
        # outputs.logits shape = [batch_size, 1] (for regression)
        preds = outputs.logits.squeeze().cpu().numpy()
        predictions.extend(preds)


100%|██████████| 1172/1172 [03:25<00:00,  5.70it/s]


In [ ]:
import pandas as pd

submission = pd.DataFrame({
    "sample_id": test_df["sample_id"],
    "price": predictions
})

submission.to_csv("/content/submission_bert.csv", index=False)
print("✅ Saved predictions to /content/submission_bert.csv")


✅ Saved predictions to /content/submission_bert.csv


In [ ]:
submission.head()


,sample_id,price
0,100179,11.457910
1,245611,7.637079
2,146263,20.708153
3,95658,8.410716
4,36806,25.443129
